In [1]:
# ============================================================
# 1. Imports, reproducibility, and device
# ============================================================

import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from configs.swin3d_small import TRAIN_CONFIG
from datasets.belize_laz_dataset import create_dataloaders
from losses.generalized_dice import GeneralizedDiceLoss

from losses.swin3d_archaeological_vicinity_loss import (
    Swin3DVicinityLossConfig,
    Swin3DSimpleVicinityLoss,
)

from models.swin3d_model import Model

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


/path/to/envs/maya_env/lib/python3.10/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


Device: cuda
GPU: Tesla V100-PCIE-32GB


In [2]:
# ============================================================
# 2. Training configuration
# ============================================================

print("Training configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")

required_keys = [
    "num_classes",
    "voxel_max",
    "batch_size",
    "augmentation_ratio",
    "max_epochs",
    "warmup_epochs",
    "early_stopping_patience",
    "learning_rate",
    "weight_decay",
]

missing_keys = [
    key for key in required_keys
    if key not in TRAIN_CONFIG
]

if missing_keys:
    raise KeyError(
        f"Missing TRAIN_CONFIG keys: {missing_keys}"
    )

if TRAIN_CONFIG["warmup_epochs"] >= TRAIN_CONFIG["max_epochs"]:
    raise ValueError(
        "warmup_epochs must be smaller than max_epochs."
    )


Training configuration:
  dataset: BelizeSlopeValue_plz_str_grd_100m_06mslope
  num_classes: 3
  input_features: ['x', 'y', 'z', 'slope']
  voxel_size: 0.2
  voxel_max: 65000
  batch_size: 16
  augmentations: ['jitter', 'flip', 'rotation']
  augmentation_ratio: 6
  trainable_layers: all
  max_epochs: 100
  warmup_epochs: 10
  early_stopping_patience: 10
  loss_function: generalized_dice
  optimizer: AdamW
  learning_rate: 3e-05
  weight_decay: 0.001
  scheduler: cosine


In [3]:
# ============================================================
# 3. Data loaders
# ============================================================

train_loader, validation_loader, test_loader = create_dataloaders(
    split_dir="dataset_splits",
    batch_size=TRAIN_CONFIG["batch_size"],
    voxel_max=TRAIN_CONFIG["voxel_max"],
    num_workers=4,
    augmentation_ratio=TRAIN_CONFIG["augmentation_ratio"],
)

print("Training samples:", len(train_loader.dataset))
print("Validation samples:", len(validation_loader.dataset))
print("Test samples:", len(test_loader.dataset))

print()
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Test batches:", len(test_loader))

print()
print(
    "Augmentation ratio:",
    TRAIN_CONFIG["augmentation_ratio"],
)


Training samples: 2226
Validation samples: 79
Test samples: 80

Training batches: 139
Validation batches: 5
Test batches: 5

Augmentation ratio: 6


In [4]:
# ============================================================
# 4. Batch and epoch helper functions
# ============================================================

def move_batch_to_device(batch, device):
    (
        coords,
        features,
        offset,
        labels,
        paths,
        metric_coords,
    ) = batch

    coords = coords.to(
        device,
        non_blocking=True,
    ).float()

    features = features.to(
        device,
        non_blocking=True,
    ).float()

    offset = offset.to(
        device,
        non_blocking=True,
    ).long()

    labels = labels.to(
        device,
        non_blocking=True,
    ).long()

    return coords, features, offset, labels


def offset_to_batch(offset):
    """
    Convert cumulative point offsets to one batch ID per point.
    """

    counts = torch.diff(
        offset,
        prepend=offset.new_zeros(1),
    )

    batch_ids = torch.arange(
        len(offset),
        device=offset.device,
        dtype=torch.long,
    )

    return torch.repeat_interleave(
        batch_ids,
        counts,
    )


def calculate_batch_accuracy(logits, labels):
    predictions = logits.argmax(dim=1)

    correct = (
        predictions == labels
    ).sum().item()

    total = labels.numel()

    return correct, total


def get_vicinity_weight(epoch_number):
    """
    Linear vicinity-loss warm-up.

    L_total = L_Dice + lambda(t) * L_vicinity
    """

    if VICINITY_WARMUP_EPOCHS <= 0:
        return VICINITY_WEIGHT

    progress = min(
        max(
            float(epoch_number)
            /
            float(VICINITY_WARMUP_EPOCHS),
            0.0,
        ),
        1.0,
    )

    return VICINITY_WEIGHT * progress


def train_one_epoch(
    model,
    loader,
    dice_criterion,
    vicinity_criterion,
    optimizer,
    device,
    epoch_number,
):
    model.train()

    vicinity_weight = get_vicinity_weight(
        epoch_number
    )

    total_loss = 0.0
    total_dice_loss = 0.0
    total_vicinity_loss = 0.0
    total_hit_loss = 0.0
    total_far_loss = 0.0

    total_correct = 0
    total_points = 0

    for batch_index, batch in enumerate(loader):

        coords, features, offset, labels = (
            move_batch_to_device(
                batch,
                device,
            )
        )

        point_batch = offset_to_batch(
            offset
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            (coords, features, offset)
        )

        if isinstance(logits, tuple):
            logits = logits[0]

        if logits.shape[0] != labels.shape[0]:
            raise RuntimeError(
                "Logit and label point counts do not match: "
                f"logits={logits.shape}, labels={labels.shape}"
            )

        dice_loss = dice_criterion(
            logits,
            labels,
        )

        (
            vicinity_loss,
            vicinity_parts,
        ) = vicinity_criterion(
            logits=logits,
            target=labels,
            coord=coords,
            batch=point_batch,
            return_components=True,
        )

        loss = (
            dice_loss
            +
            vicinity_weight * vicinity_loss
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss in batch {batch_index}: "
                f"{loss.item()}"
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        correct, number_of_points = (
            calculate_batch_accuracy(
                logits,
                labels,
            )
        )

        total_loss += (
            loss.item()
            * number_of_points
        )

        total_dice_loss += (
            dice_loss.item()
            * number_of_points
        )

        total_vicinity_loss += (
            vicinity_loss.item()
            * number_of_points
        )

        total_hit_loss += (
            vicinity_parts[
                "vicinity_hit"
            ].item()
            * number_of_points
        )

        total_far_loss += (
            vicinity_parts[
                "vicinity_far_negative"
            ].item()
            * number_of_points
        )

        total_correct += correct
        total_points += number_of_points

        if (
            batch_index == 0
            or (batch_index + 1) % 10 == 0
            or batch_index + 1 == len(loader)
        ):
            print(
                f"  Batch {batch_index + 1:04d}/"
                f"{len(loader):04d} | "
                f"Total: {loss.item():.6f} | "
                f"Dice: {dice_loss.item():.6f} | "
                f"Vicinity: {vicinity_loss.item():.6f} | "
                f"Hit: {vicinity_parts['vicinity_hit'].item():.6f} | "
                f"Far: {vicinity_parts['vicinity_far_negative'].item():.6f} | "
                f"λ: {vicinity_weight:.4f}"
            )

    denominator = max(
        total_points,
        1,
    )

    average_loss = (
        total_loss
        /
        denominator
    )

    average_dice_loss = (
        total_dice_loss
        /
        denominator
    )

    average_vicinity_loss = (
        total_vicinity_loss
        /
        denominator
    )

    average_hit_loss = (
        total_hit_loss
        /
        denominator
    )

    average_far_loss = (
        total_far_loss
        /
        denominator
    )

    accuracy = (
        total_correct
        /
        denominator
    )

    return (
        average_loss,
        average_dice_loss,
        average_vicinity_loss,
        average_hit_loss,
        average_far_loss,
        vicinity_weight,
        accuracy,
    )


@torch.no_grad()
def evaluate(
    model,
    loader,
    dice_criterion,
    vicinity_criterion,
    device,
    epoch_number,
):
    model.eval()

    vicinity_weight = get_vicinity_weight(
        epoch_number
    )

    total_loss = 0.0
    total_dice_loss = 0.0
    total_vicinity_loss = 0.0
    total_hit_loss = 0.0
    total_far_loss = 0.0

    total_correct = 0
    total_points = 0

    for batch in loader:

        coords, features, offset, labels = (
            move_batch_to_device(
                batch,
                device,
            )
        )

        point_batch = offset_to_batch(
            offset
        )

        logits = model(
            (coords, features, offset)
        )

        if isinstance(logits, tuple):
            logits = logits[0]

        if logits.shape[0] != labels.shape[0]:
            raise RuntimeError(
                "Logit and label point counts do not match: "
                f"logits={logits.shape}, labels={labels.shape}"
            )

        dice_loss = dice_criterion(
            logits,
            labels,
        )

        (
            vicinity_loss,
            vicinity_parts,
        ) = vicinity_criterion(
            logits=logits,
            target=labels,
            coord=coords,
            batch=point_batch,
            return_components=True,
        )

        loss = (
            dice_loss
            +
            vicinity_weight * vicinity_loss
        )

        correct, number_of_points = (
            calculate_batch_accuracy(
                logits,
                labels,
            )
        )

        total_loss += (
            loss.item()
            * number_of_points
        )

        total_dice_loss += (
            dice_loss.item()
            * number_of_points
        )

        total_vicinity_loss += (
            vicinity_loss.item()
            * number_of_points
        )

        total_hit_loss += (
            vicinity_parts[
                "vicinity_hit"
            ].item()
            * number_of_points
        )

        total_far_loss += (
            vicinity_parts[
                "vicinity_far_negative"
            ].item()
            * number_of_points
        )

        total_correct += correct
        total_points += number_of_points


    denominator = max(
        total_points,
        1,
    )

    average_loss = (
        total_loss
        /
        denominator
    )

    average_dice_loss = (
        total_dice_loss
        /
        denominator
    )

    average_vicinity_loss = (
        total_vicinity_loss
        /
        denominator
    )

    average_hit_loss = (
        total_hit_loss
        /
        denominator
    )

    average_far_loss = (
        total_far_loss
        /
        denominator
    )

    accuracy = (
        total_correct
        /
        denominator
    )

    return (
        average_loss,
        average_dice_loss,
        average_vicinity_loss,
        average_hit_loss,
        average_far_loss,
        accuracy,
    )


In [5]:
# ============================================================
# 5. Model, losses, optimizer, and scheduler
# ============================================================

model = Model(
    num_classes=TRAIN_CONFIG["num_classes"],
    in_channels=4,
).to(device)

for parameter in model.parameters():
    parameter.requires_grad = True


# ------------------------------------------------------------
# Loss functions
#
# Project class mapping:
#   0 = Ground
#   1 = Structure
#   2 = Plazuela
#
# Dataset "coord" is already local XYZ in meters,
# therefore coordinate_scale_m = 1.0.
# ------------------------------------------------------------

dice_criterion = GeneralizedDiceLoss(
    num_classes=TRAIN_CONFIG["num_classes"],
)

vicinity_config = Swin3DVicinityLossConfig(
    ground_label=0,
    structure_label=1,
    plazuela_label=2,
    ignore_label=-100,

    vicinity_radius_m=2.0,
    grid_size_m=0.50,

    coordinate_scale_m=1.0,
    horizontal_axes=(0, 1),

    far_negative_weight=0.50,
    focal_gamma=2.0,

    max_grid_cells=1_000_000,
)

vicinity_criterion = Swin3DSimpleVicinityLoss(
    vicinity_config
).to(device)


VICINITY_WEIGHT = 0.10
VICINITY_WARMUP_EPOCHS = 10


print(
    "Combined training loss = "
    "Dice + λ(epoch) * Vicinity"
)

print(
    "Maximum vicinity weight:",
    VICINITY_WEIGHT,
)

print(
    "Vicinity warm-up epochs:",
    VICINITY_WARMUP_EPOCHS,
)

print(
    "Vicinity radius:",
    vicinity_config.vicinity_radius_m,
    "m",
)

print(
    "Vicinity grid size:",
    vicinity_config.grid_size_m,
    "m",
)

print(
    "Far-negative weight:",
    vicinity_config.far_negative_weight,
)

print(
    "Focal gamma:",
    vicinity_config.focal_gamma,
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
)


# ------------------------------------------------------------
# 10-epoch warmup + cosine decay
# ------------------------------------------------------------

warmup_epochs = TRAIN_CONFIG["warmup_epochs"]
max_epochs = TRAIN_CONFIG["max_epochs"]

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_epochs,
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max_epochs - warmup_epochs,
    eta_min=1e-7,
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        warmup_scheduler,
        cosine_scheduler,
    ],
    milestones=[
        warmup_epochs,
    ],
)


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print()
print(
    "Dice loss:",
    dice_criterion.__class__.__name__,
)

print(
    "Vicinity loss:",
    vicinity_criterion.__class__.__name__,
)

print(
    "Optimizer:",
    optimizer.__class__.__name__,
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}",
)

print(
    "Base learning rate:",
    TRAIN_CONFIG["learning_rate"],
)

print(
    "Initial warm-up LR:",
    optimizer.param_groups[0]["lr"],
)

print(
    "Warm-up epochs:",
    warmup_epochs,
)

print(
    "Cosine epochs:",
    max_epochs - warmup_epochs,
)


Combined training loss = Dice + λ(epoch) * Vicinity
Maximum vicinity weight: 0.1
Vicinity warm-up epochs: 10
Vicinity radius: 2.0 m
Vicinity grid size: 0.5 m
Far-negative weight: 0.5
Focal gamma: 2.0

Dice loss: GeneralizedDiceLoss
Vicinity loss: Swin3DSimpleVicinityLoss
Optimizer: AdamW
Trainable parameters: 24,764,691
Base learning rate: 3e-05
Initial warm-up LR: 3e-06
Warm-up epochs: 10
Cosine epochs: 90


In [6]:
# ============================================================
# 6. One-batch Vicinity Loss sanity check
# ============================================================

batch = next(iter(train_loader))

coords, features, offset, labels = move_batch_to_device(
    batch,
    device,
)

point_batch = offset_to_batch(
    offset
)

print("Coordinates:", coords.shape)
print("Features:", features.shape)
print("Offsets:", offset.shape)
print("Labels:", labels.shape)
print("Point batch:", point_batch.shape)

print(
    "Input channels:",
    coords.shape[1] + features.shape[1],
)

model.train()

optimizer.zero_grad(
    set_to_none=True
)

logits = model(
    (coords, features, offset)
)

if isinstance(logits, tuple):
    logits = logits[0]

print("Logits:", logits.shape)

if logits.shape[0] != labels.shape[0]:
    raise RuntimeError(
        "Sanity check failed: model output and labels "
        "have different point counts."
    )

if logits.shape[1] != TRAIN_CONFIG["num_classes"]:
    raise RuntimeError(
        "Sanity check failed: wrong number of classes."
    )


dice_loss = dice_criterion(
    logits,
    labels,
)

(
    vicinity_loss,
    vicinity_parts,
) = vicinity_criterion(
    logits=logits,
    target=labels,
    coord=coords,
    batch=point_batch,
    return_components=True,
)

sanity_weight = get_vicinity_weight(
    epoch_number=1
)

combined_loss = (
    dice_loss
    +
    sanity_weight * vicinity_loss
)

print()

print(
    f"Dice loss:         "
    f"{dice_loss.item():.6f}"
)

print(
    f"Vicinity loss:     "
    f"{vicinity_loss.item():.6f}"
)

print(
    f"  Hit loss:        "
    f"{vicinity_parts['vicinity_hit'].item():.6f}"
)

print(
    f"  Far-neg loss:    "
    f"{vicinity_parts['vicinity_far_negative'].item():.6f}"
)

print(
    f"Vicinity weight:   "
    f"{sanity_weight:.4f}"
)

print(
    f"Combined loss:     "
    f"{combined_loss.item():.6f}"
)


if not torch.isfinite(
    combined_loss
):
    raise RuntimeError(
        "Combined loss is not finite."
    )

combined_loss.backward()

gradient_norm = 0.0

for parameter in model.parameters():
    if parameter.grad is not None:
        gradient_norm += (
            parameter.grad.norm().item()
        )

print(
    f"Gradient norm:     "
    f"{gradient_norm:.6f}"
)

if gradient_norm <= 0:
    raise RuntimeError(
        "Vicinity sanity check failed: no gradients."
    )

optimizer.zero_grad(
    set_to_none=True
)

print()
print(
    "Vicinity Loss sanity check passed."
)


Coordinates: torch.Size([1040000, 3])
Features: torch.Size([1040000, 1])
Offsets: torch.Size([16])
Labels: torch.Size([1040000])
Point batch: torch.Size([1040000])
Input channels: 4


/path/to/envs/maya_env/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/path/to/software/Swin3D/Swin3D/modules/swin3d_layers.py:239: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:78.)
  offset = torch.cuda.IntTensor(offset)


Logits: torch.Size([1040000, 3])

Dice loss:         0.976864
Vicinity loss:     0.615202
  Hit loss:        0.353265
  Far-neg loss:    0.523872
Vicinity weight:   0.0100
Combined loss:     0.983016
Gradient norm:     0.552943

Vicinity Loss sanity check passed.


In [7]:
# ============================================================
# 7. Checkpoint paths
# ============================================================

checkpoint_dir = Path(
    "/path/to/project_storage/"
    "swin3d_checkpoints/"
    "dice_vicinity"
)

checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)


best_checkpoint_path = (
    checkpoint_dir /
    "swin3d_dice_vicinity_best_02.pth"
)

last_checkpoint_path = (
    checkpoint_dir /
    "swin3d_dice_vicinity_last_02.pth"
)

RESUME_TRAINING = True

print(
    "Best checkpoint:",
    best_checkpoint_path,
)

print(
    "Last checkpoint:",
    last_checkpoint_path,
)


Best checkpoint: /path/to/project_storage/swin3d_checkpoints/dice_vicinity/swin3d_dice_vicinity_best_02.pth
Last checkpoint: /path/to/project_storage/swin3d_checkpoints/dice_vicinity/swin3d_dice_vicinity_last_02.pth


In [10]:
# ============================================================
# 8. Train with Dice + Vicinity Loss
# ============================================================

patience = TRAIN_CONFIG["early_stopping_patience"]

start_epoch = 0
best_validation_dice_loss = float("inf")
epochs_without_improvement = 0

training_history = {
    "train_loss": [],
    "train_dice_loss": [],
    "train_vicinity_loss": [],
    "train_vicinity_hit_loss": [],
    "train_vicinity_far_loss": [],
    "vicinity_weight": [],
    "train_accuracy": [],

    "validation_loss": [],
    "validation_dice_loss": [],
    "validation_vicinity_loss": [],
    "validation_vicinity_hit_loss": [],
    "validation_vicinity_far_loss": [],
    "validation_accuracy": [],

    "learning_rate": [],
}


if (
    RESUME_TRAINING
    and
    last_checkpoint_path.exists()
):

    print(
        "Loading previous Dice + Vicinity checkpoint..."
    )

    checkpoint = torch.load(
        last_checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        checkpoint["scheduler_state_dict"]
    )

    start_epoch = checkpoint["epoch"]

    best_validation_dice_loss = checkpoint.get(
        "best_validation_dice_loss",
        float("inf"),
    )

    epochs_without_improvement = checkpoint.get(
        "epochs_without_improvement",
        0,
    )

    training_history = checkpoint.get(
        "training_history",
        training_history,
    )

    print(
        f"Last completed epoch: "
        f"{start_epoch}"
    )

    print(
        f"Resuming from epoch: "
        f"{start_epoch + 1}"
    )

    print(
        "Current LR:",
        optimizer.param_groups[0]["lr"],
    )

else:

    print(
        "Starting new Dice + Vicinity training."
    )


for epoch in range(
    start_epoch,
    max_epochs,
):

    epoch_number = epoch + 1

    current_learning_rate = (
        optimizer.param_groups[0]["lr"]
    )

    print(
        f"\n{'=' * 65}\n"
        f"Epoch {epoch_number}/{max_epochs} | "
        f"LR: {current_learning_rate:.8f}\n"
        f"{'=' * 65}"
    )


    # --------------------------------------------------------
    # Training: Dice + Vicinity
    # --------------------------------------------------------

    (
        train_loss,
        train_dice_loss,
        train_vicinity_loss,
        train_vicinity_hit_loss,
        train_vicinity_far_loss,
        current_vicinity_weight,
        train_accuracy,
    ) = train_one_epoch(
        model=model,
        loader=train_loader,
        dice_criterion=dice_criterion,
        vicinity_criterion=vicinity_criterion,
        optimizer=optimizer,
        device=device,
        epoch_number=epoch_number,
    )


    # --------------------------------------------------------
    # Validation
    #
    # Combined validation loss is reported for comparison.
    # Best model / early stopping uses validation Dice loss.
    # --------------------------------------------------------

    (
        validation_loss,
        validation_dice_loss,
        validation_vicinity_loss,
        validation_vicinity_hit_loss,
        validation_vicinity_far_loss,
        validation_accuracy,
    ) = evaluate(
        model=model,
        loader=validation_loader,
        dice_criterion=dice_criterion,
        vicinity_criterion=vicinity_criterion,
        device=device,
        epoch_number=epoch_number,
    )


    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    training_history[
        "train_loss"
    ].append(
        train_loss
    )

    training_history[
        "train_dice_loss"
    ].append(
        train_dice_loss
    )

    training_history[
        "train_vicinity_loss"
    ].append(
        train_vicinity_loss
    )

    training_history[
        "train_vicinity_hit_loss"
    ].append(
        train_vicinity_hit_loss
    )

    training_history[
        "train_vicinity_far_loss"
    ].append(
        train_vicinity_far_loss
    )

    training_history[
        "vicinity_weight"
    ].append(
        current_vicinity_weight
    )

    training_history[
        "train_accuracy"
    ].append(
        train_accuracy
    )

    training_history[
        "validation_loss"
    ].append(
        validation_loss
    )

    training_history[
        "validation_dice_loss"
    ].append(
        validation_dice_loss
    )

    training_history[
        "validation_vicinity_loss"
    ].append(
        validation_vicinity_loss
    )

    training_history[
        "validation_vicinity_hit_loss"
    ].append(
        validation_vicinity_hit_loss
    )

    training_history[
        "validation_vicinity_far_loss"
    ].append(
        validation_vicinity_far_loss
    )

    training_history[
        "validation_accuracy"
    ].append(
        validation_accuracy
    )

    training_history[
        "learning_rate"
    ].append(
        current_learning_rate
    )


    print(
        f"\nEpoch {epoch_number:03d} results\n"
        f"Train total loss:        {train_loss:.6f}\n"
        f"Train Dice loss:         {train_dice_loss:.6f}\n"
        f"Train Vicinity:          {train_vicinity_loss:.6f}\n"
        f"  Train Hit:             {train_vicinity_hit_loss:.6f}\n"
        f"  Train Far-negative:    {train_vicinity_far_loss:.6f}\n"
        f"Vicinity weight:         {current_vicinity_weight:.4f}\n"
        f"Train accuracy:          {train_accuracy:.4f}\n"
        f"\n"
        f"Validation total:        {validation_loss:.6f}\n"
        f"Validation Dice:         {validation_dice_loss:.6f}\n"
        f"Validation Vicinity:     {validation_vicinity_loss:.6f}\n"
        f"  Validation Hit:        {validation_vicinity_hit_loss:.6f}\n"
        f"  Validation Far-neg:    {validation_vicinity_far_loss:.6f}\n"
        f"Validation accuracy:     {validation_accuracy:.4f}\n"
        f"Learning rate:           {current_learning_rate:.8f}"
    )


    # --------------------------------------------------------
    # Best model based on validation Dice loss
    # --------------------------------------------------------

    improved = (
        validation_dice_loss
        <
        best_validation_dice_loss
    )

    if improved:

        best_validation_dice_loss = (
            validation_dice_loss
        )

        epochs_without_improvement = 0

        print(
            "New best validation Dice loss: "
            f"{best_validation_dice_loss:.6f}"
        )

    elif epoch_number > warmup_epochs:

        epochs_without_improvement += 1

        print(
            "No validation improvement: "
            f"{epochs_without_improvement}/"
            f"{patience}"
        )

    else:

        print(
            "No validation improvement. "
            "Early stopping is disabled during warm-up."
        )


    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------

    scheduler.step()


    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    checkpoint = {
        "epoch": epoch_number,

        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),

        # Training
        "train_loss": train_loss,
        "train_dice_loss": train_dice_loss,
        "train_vicinity_loss": train_vicinity_loss,
        "train_vicinity_hit_loss": train_vicinity_hit_loss,
        "train_vicinity_far_loss": train_vicinity_far_loss,
        "vicinity_weight_current": current_vicinity_weight,
        "train_accuracy": train_accuracy,

        # Validation
        "validation_loss": validation_loss,
        "validation_dice_loss": validation_dice_loss,
        "validation_vicinity_loss": validation_vicinity_loss,
        "validation_vicinity_hit_loss": validation_vicinity_hit_loss,
        "validation_vicinity_far_loss": validation_vicinity_far_loss,
        "validation_accuracy": validation_accuracy,

        # Best model / early stopping
        "best_validation_dice_loss": (
            best_validation_dice_loss
        ),

        "epochs_without_improvement": (
            epochs_without_improvement
        ),

        # Loss configuration
        "vicinity_weight_max": VICINITY_WEIGHT,
        "vicinity_warmup_epochs": VICINITY_WARMUP_EPOCHS,

        "vicinity_config": {
            "ground_label": vicinity_config.ground_label,
            "structure_label": vicinity_config.structure_label,
            "plazuela_label": vicinity_config.plazuela_label,
            "ignore_label": vicinity_config.ignore_label,
            "vicinity_radius_m": vicinity_config.vicinity_radius_m,
            "grid_size_m": vicinity_config.grid_size_m,
            "coordinate_scale_m": vicinity_config.coordinate_scale_m,
            "horizontal_axes": vicinity_config.horizontal_axes,
            "far_negative_weight": vicinity_config.far_negative_weight,
            "focal_gamma": vicinity_config.focal_gamma,
            "max_grid_cells": vicinity_config.max_grid_cells,
        },

        "validation_loss_type": (
            "dice_plus_vicinity_reported_"
            "best_checkpoint_by_dice"
        ),

        "training_history": training_history,
        "config": TRAIN_CONFIG,
    }


    torch.save(
        checkpoint,
        last_checkpoint_path,
    )


    if improved:

        torch.save(
            checkpoint,
            best_checkpoint_path,
        )

        print(
            "Saved best model:",
            best_checkpoint_path,
        )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        epoch_number > warmup_epochs
        and
        epochs_without_improvement >= patience
    ):

        print(
            "Best validation Dice loss:",
            f"{best_validation_dice_loss:.6f}",
        )

        break


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


Starting new Dice + Vicinity training.

Epoch 1/100 | LR: 0.00000300
  Batch 0001/0139 | Total: 0.970351 | Dice: 0.964204 | Vicinity: 0.614701 | Hit: 0.353042 | Far: 0.523317 | λ: 0.0100
  Batch 0010/0139 | Total: 0.982559 | Dice: 0.976618 | Vicinity: 0.594039 | Hit: 0.333339 | Far: 0.521398 | λ: 0.0100
  Batch 0020/0139 | Total: 0.999951 | Dice: 0.994079 | Vicinity: 0.587191 | Hit: 0.325136 | Far: 0.524110 | λ: 0.0100
  Batch 0030/0139 | Total: 0.974836 | Dice: 0.969044 | Vicinity: 0.579158 | Hit: 0.319946 | Far: 0.518426 | λ: 0.0100
  Batch 0040/0139 | Total: 0.995098 | Dice: 0.989470 | Vicinity: 0.562839 | Hit: 0.301793 | Far: 0.522092 | λ: 0.0100
  Batch 0050/0139 | Total: 0.991101 | Dice: 0.985415 | Vicinity: 0.568618 | Hit: 0.308953 | Far: 0.519330 | λ: 0.0100
  Batch 0060/0139 | Total: 0.994284 | Dice: 0.988766 | Vicinity: 0.551829 | Hit: 0.291096 | Far: 0.521466 | λ: 0.0100
  Batch 0070/0139 | Total: 0.955466 | Dice: 0.950034 | Vicinity: 0.543185 | Hit: 0.285477 | Far: 0.515416

/path/to/envs/maya_env/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)



Epoch 11/100 | LR: 0.00003000
  Batch 0001/0139 | Total: 0.919005 | Dice: 0.898956 | Vicinity: 0.200494 | Hit: 0.084465 | Far: 0.232057 | λ: 0.1000
  Batch 0010/0139 | Total: 0.892787 | Dice: 0.872821 | Vicinity: 0.199658 | Hit: 0.048436 | Far: 0.302445 | λ: 0.1000
  Batch 0020/0139 | Total: 0.975125 | Dice: 0.953702 | Vicinity: 0.214234 | Hit: 0.060401 | Far: 0.307667 | λ: 0.1000
  Batch 0030/0139 | Total: 0.984618 | Dice: 0.965889 | Vicinity: 0.187283 | Hit: 0.045616 | Far: 0.283333 | λ: 0.1000
  Batch 0040/0139 | Total: 0.990971 | Dice: 0.970149 | Vicinity: 0.208219 | Hit: 0.049591 | Far: 0.317256 | λ: 0.1000
  Batch 0050/0139 | Total: 0.988553 | Dice: 0.968703 | Vicinity: 0.198499 | Hit: 0.079584 | Far: 0.237829 | λ: 0.1000
  Batch 0060/0139 | Total: 0.984395 | Dice: 0.965626 | Vicinity: 0.187697 | Hit: 0.041819 | Far: 0.291756 | λ: 0.1000
  Batch 0070/0139 | Total: 1.001571 | Dice: 0.985515 | Vicinity: 0.160553 | Hit: 0.031457 | Far: 0.258192 | λ: 0.1000
  Batch 0080/0139 | Total

In [11]:
# ============================================================
# 9. Plot training history
# ============================================================

epochs = np.arange(
    1,
    len(training_history["train_loss"]) + 1,
)


# Combined training and validation loss
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    training_history["train_loss"],
    label="Train Combined Loss",
)

plt.plot(
    epochs,
    training_history["validation_loss"],
    label="Validation Combined Loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Combined Training and Validation Loss"
)

plt.legend()
plt.grid(True)
plt.show()


# Dice loss
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    training_history["train_dice_loss"],
    label="Train Dice",
)

plt.plot(
    epochs,
    training_history["validation_dice_loss"],
    label="Validation Dice",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Generalized Dice Loss"
)

plt.legend()
plt.grid(True)
plt.show()


# Vicinity loss
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    training_history["train_vicinity_loss"],
    label="Train Vicinity",
)

plt.plot(
    epochs,
    training_history["validation_vicinity_loss"],
    label="Validation Vicinity",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Vicinity Loss"
)

plt.legend()
plt.grid(True)
plt.show()


# Vicinity components
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    training_history["train_vicinity_hit_loss"],
    label="Train Hit",
)

plt.plot(
    epochs,
    training_history["train_vicinity_far_loss"],
    label="Train Far-negative",
)

plt.plot(
    epochs,
    training_history["validation_vicinity_hit_loss"],
    label="Validation Hit",
)

plt.plot(
    epochs,
    training_history["validation_vicinity_far_loss"],
    label="Validation Far-negative",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Vicinity Loss Components"
)

plt.legend()
plt.grid(True)
plt.show()


# Accuracy
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    training_history["train_accuracy"],
    label="Train",
)

plt.plot(
    epochs,
    training_history["validation_accuracy"],
    label="Validation",
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "Training and Validation Accuracy"
)

plt.legend()
plt.grid(True)
plt.show()


# Vicinity weight
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    training_history["vicinity_weight"],
)

plt.axvline(
    VICINITY_WARMUP_EPOCHS,
    linestyle="--",
    label="Full vicinity weight",
)

plt.xlabel("Epoch")
plt.ylabel("Vicinity Weight")

plt.title(
    "Vicinity Loss Warm-up"
)

plt.legend()
plt.grid(True)
plt.show()


# Learning rate
plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    training_history["learning_rate"],
)

plt.axvline(
    warmup_epochs,
    linestyle="--",
    label="End of warm-up",
)

plt.xlabel("Epoch")
plt.ylabel("Learning Rate")

plt.title(
    "Learning Rate Schedule"
)

plt.legend()
plt.grid(True)
plt.show()


SyntaxError: unterminated string literal (detected at line 49) (1035014663.py, line 49)

In [ ]:
# ============================================================
# 10. Load best checkpoint and evaluate test set
# ============================================================

best_checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

print(
    "Loaded best checkpoint from epoch:",
    best_checkpoint["epoch"],
)

print(
    "Best validation Dice loss:",
    best_checkpoint[
        "validation_dice_loss"
    ],
)

print(
    "Validation combined loss:",
    best_checkpoint[
        "validation_loss"
    ],
)

print(
    "Validation Vicinity loss:",
    best_checkpoint[
        "validation_vicinity_loss"
    ],
)

print(
    "Maximum vicinity weight:",
    best_checkpoint[
        "vicinity_weight_max"
    ],
)


(
    test_loss,
    test_dice_loss,
    test_vicinity_loss,
    test_vicinity_hit_loss,
    test_vicinity_far_loss,
    test_accuracy,
) = evaluate(
    model=model,
    loader=test_loader,
    dice_criterion=dice_criterion,
    vicinity_criterion=vicinity_criterion,
    device=device,
    epoch_number=best_checkpoint["epoch"],
)


print()
print("Final test results")
print("------------------")

print(
    f"Test combined loss:      "
    f"{test_loss:.6f}"
)

print(
    f"Test Dice loss:          "
    f"{test_dice_loss:.6f}"
)

print(
    f"Test Vicinity loss:      "
    f"{test_vicinity_loss:.6f}"
)

print(
    f"  Test Hit loss:         "
    f"{test_vicinity_hit_loss:.6f}"
)

print(
    f"  Test Far-negative:     "
    f"{test_vicinity_far_loss:.6f}"
)

print(
    f"Test accuracy:           "
    f"{test_accuracy:.4f}"
)
